# Train TRADES on Nasdaq ITCH days (INTC)

Runs on a free Kaggle or Colab GPU. It needs the dataset `trades_bundle.zip` (order-book data for eight days, a patch, and the training script).

**Before running:** in the right-hand panel set *Accelerator* to **GPU T4** and *Internet* to **On**, and add the `trades-bundle` dataset to the notebook.

The last cell leaves the trained checkpoint in the notebook's output folder. Download `trades_checkpoint.zip` from there.

In [ ]:
import glob, os, shutil, subprocess, sys

# Find the bundle wherever the platform unpacked it (Kaggle: /kaggle/input/..., Colab: /content/...).
hits = [p for base in ("/kaggle/input", "/content", ".") for p in glob.glob(f"{base}/**/deepmarket_simbench.patch", recursive=True)]
assert hits, "Bundle not found: add the trades-bundle dataset to this notebook (or unzip trades_bundle.zip under /content)."
BUNDLE = os.path.dirname(hits[0])
# Work outside the output folder: only the final zip is written to OUT, so the saved notebook output stays small.
OUT = "/kaggle/working" if os.path.isdir("/kaggle/working") else "/content"
WORK = "/tmp/trades_work"
os.makedirs(WORK, exist_ok=True)
print("bundle:", BUNDLE, "| working dir:", WORK, "| output dir:", OUT)
print(os.listdir(BUNDLE))
import torch
print("GPU available:", torch.cuda.is_available(), torch.cuda.get_device_name(0) if torch.cuda.is_available() else "")

In [ ]:
# DeepMarket at the commit the patch was made for, plus its dependencies.
os.chdir(WORK)
if not os.path.isdir("DeepMarket"):
    subprocess.run("git clone -q https://github.com/LeonardoBerti00/DeepMarket.git", shell=True, check=True, env={**os.environ, "GIT_LFS_SKIP_SMUDGE": "1"})
os.chdir(f"{WORK}/DeepMarket")
subprocess.run("git checkout -q 8f1f89b7285c79a73f528b88b60f74ce58faadc4 && git checkout -q -- . ", shell=True, check=True, env={**os.environ, "GIT_LFS_SKIP_SMUDGE": "1"})
subprocess.run(f"git apply --whitespace=nowarn {BUNDLE}/deepmarket_simbench.patch", shell=True, check=True)
subprocess.run(f"{sys.executable} -m pip install -q lightning einops torch_ema lion_pytorch wandb", shell=True, check=True)
shutil.copy(f"{BUNDLE}/train_trades_multiday.py", "train_trades_multiday.py")
print("DeepMarket ready")

In [ ]:
# Put the order-book data where DeepMarket expects it.
src = [d for d in glob.glob(f"{BUNDLE}/**/INTC_*_*", recursive=True) if os.path.isdir(d)][0]
dst = f"{WORK}/DeepMarket/data/INTC/{os.path.basename(src)}"
shutil.rmtree(f"{WORK}/DeepMarket/data/INTC", ignore_errors=True)
os.makedirs(os.path.dirname(dst), exist_ok=True)
shutil.copytree(src, dst)
print(sorted(os.listdir(dst)))

In [ ]:
# Train. MINUTES must leave time for the notebook to finish inside the platform's 12-hour session limit.
MINUTES = 600
os.chdir(f"{WORK}/DeepMarket")
p = subprocess.Popen(f"{sys.executable} -u train_trades_multiday.py --symbol INTC --split 5,1,2 --minutes {MINUTES}", shell=True,
                     stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)
with open(f"{OUT}/train_log.txt", "w") as log:
    for line in p.stdout:
        if "Warning" in line or "warn" in line:
            continue
        print(line, end="")
        log.write(line)
        log.flush()
print("exit code", p.wait())

In [ ]:
# Package what the simulation needs: the checkpoint, the normalised-data statistics inputs, and the log.
os.chdir(f"{WORK}/DeepMarket")
out = f"{WORK}/trades_checkpoint"
shutil.rmtree(out, ignore_errors=True)
os.makedirs(out)
for f in glob.glob("data/checkpoints/TRADES/*.ckpt"):
    shutil.copy(f, out)
shutil.copy("constants.py", out)
shutil.copy(f"{OUT}/train_log.txt", out)
shutil.make_archive(f"{OUT}/trades_checkpoint", "zip", out)
print(os.listdir(out))
print("Download:", f"{OUT}/trades_checkpoint.zip")